# Notebook 09 — Kaggle End-to-End Submission

Self-contained notebook for the ROGII Wellbore Geology Prediction competition.

## Final recipe (from Notebooks 06–08 / 07B)

| Piece | Value |
|-------|-------|
| Models | Extra Trees + XGBoost |
| Ensemble weights | **0.8** Extra Trees + **0.2** XGBoost |
| Features | `MD`, `GR`, `TVT_input`, `X`, `Y`, `Z` |
| Prediction rows | Only where `TVT_input` is missing |
| Metric | RMSE |

No project `src` imports. Utilities are inlined below so this notebook runs on Kaggle and locally.


## 1. Imports


In [ ]:
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor


warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Imports ready.")


## 2. Configure Paths (Kaggle + Local)


In [ ]:
def list_kaggle_input_tree(root: Path, max_depth: int = 4) -> None:
    """Print a shallow tree for debugging path issues."""
    print(f"Contents of {root}:")
    if not root.exists():
        print("  (path does not exist)")
        return

    for path in sorted(root.rglob("*")):
        try:
            relative = path.relative_to(root)
        except ValueError:
            continue
        if len(relative.parts) > max_depth:
            continue
        marker = "/" if path.is_dir() else ""
        print(f"  {relative.as_posix()}{marker}")


def count_horizontal_csvs(directory: Path) -> int:
    if not directory.is_dir():
        return 0
    return len(list(directory.rglob("*__horizontal_well.csv")))


def is_well_data_dir(directory: Path) -> bool:
    """True if this folder (or a child) contains horizontal-well CSVs."""
    return count_horizontal_csvs(directory) > 0


def try_pair_from_root(data_root: Path):
    """Return (data_root, train_dir, test_dir) when both sides exist."""
    train_dir = data_root / "train"
    test_dir = data_root / "test"
    if train_dir.is_dir() and test_dir.is_dir():
        return data_root, train_dir, test_dir
    return None


def find_train_test_dirs(kaggle_input: Path):
    """
    Locate competition train/ and test/ under /kaggle/input.

    Handles:
      /kaggle/input/rogii-wellbore-geology-prediction/{train,test}
      /kaggle/input/competitions/rogii-wellbore-geology-prediction/{train,test}
    """
    competition_slug = "rogii-wellbore-geology-prediction"

    explicit_roots = [
        kaggle_input / competition_slug,
        kaggle_input / "competitions" / competition_slug,
    ]

    if kaggle_input.is_dir():
        for child in sorted(kaggle_input.iterdir()):
            if child.is_dir():
                explicit_roots.append(child)
                explicit_roots.append(child / competition_slug)

    seen = set()
    for root in explicit_roots:
        if not root.exists():
            continue
        key = str(root.resolve())
        if key in seen:
            continue
        seen.add(key)
        pair = try_pair_from_root(root)
        if pair is None:
            continue
        data_root, train_dir, test_dir = pair
        if is_well_data_dir(train_dir) and is_well_data_dir(test_dir):
            return data_root, train_dir, test_dir

    candidates = []
    for train_dir in sorted(kaggle_input.rglob("train")):
        if not train_dir.is_dir():
            continue
        test_dir = train_dir.parent / "test"
        if not test_dir.is_dir():
            continue
        if is_well_data_dir(train_dir) and is_well_data_dir(test_dir):
            candidates.append((train_dir.parent, train_dir, test_dir))

    if candidates:
        candidates.sort(key=lambda item: (len(item[0].parts), str(item[0])))
        return candidates[0]

    list_kaggle_input_tree(kaggle_input)
    raise FileNotFoundError(
        "Could not locate train/ and test/ under /kaggle/input. "
        "Expected something like:\n"
        f"  /kaggle/input/{competition_slug}/train\n"
        f"  /kaggle/input/{competition_slug}/test\n"
        "Re-upload this notebook after attaching the competition data, "
        "then Run All again."
    )


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "data" / "raw" / "train").is_dir():
        return cwd
    if (cwd.parent / "data" / "raw" / "train").is_dir():
        return cwd.parent
    return cwd.parent


KAGGLE_INPUT = Path("/kaggle/input")
IS_KAGGLE = KAGGLE_INPUT.exists()

if IS_KAGGLE:
    print("=== Kaggle input tree ===")
    list_kaggle_input_tree(KAGGLE_INPUT)
    print("=== Resolving train/test ===")
    DATA_ROOT, TRAIN_DIR, TEST_DIR = find_train_test_dirs(KAGGLE_INPUT)
    OUT_DIR = Path("/kaggle/working")
else:
    PROJECT_ROOT = resolve_project_root()
    DATA_ROOT = PROJECT_ROOT / "data" / "raw"
    TRAIN_DIR = DATA_ROOT / "train"
    TEST_DIR = DATA_ROOT / "test"
    OUT_DIR = PROJECT_ROOT / "submissions"

OUT_DIR.mkdir(parents=True, exist_ok=True)

assert TRAIN_DIR.is_dir(), f"Training directory not found: {TRAIN_DIR}"
assert TEST_DIR.is_dir(), f"Test directory not found: {TEST_DIR}"

train_horizontal_count = count_horizontal_csvs(TRAIN_DIR)
test_horizontal_count = count_horizontal_csvs(TEST_DIR)

assert train_horizontal_count > 0, (
    f"No *__horizontal_well.csv files found under {TRAIN_DIR}"
)
assert test_horizontal_count > 0, (
    f"No *__horizontal_well.csv files found under {TEST_DIR}"
)

print("Running on Kaggle:", IS_KAGGLE)
print("Data root:", DATA_ROOT)
print("Train dir:", TRAIN_DIR)
print("Test dir:", TEST_DIR)
print("Output dir:", OUT_DIR)
print("Train horizontal files:", train_horizontal_count)
print("Test horizontal files:", test_horizontal_count)


## 3. Inline Utilities


In [ ]:
GROUP_COLUMN = "well_id"


def discover_training_wells(train_dir: Path) -> list[str]:
    typewell_ids = {
        file.name.replace("__typewell.csv", "")
        for file in train_dir.rglob("*__typewell.csv")
    }
    horizontal_ids = {
        file.name.replace("__horizontal_well.csv", "")
        for file in train_dir.rglob("*__horizontal_well.csv")
    }
    return sorted(typewell_ids.intersection(horizontal_ids))


def discover_test_well_pairs(test_dir: Path) -> list[str]:
    typewell_ids = {
        file.name.replace("__typewell.csv", "")
        for file in test_dir.rglob("*__typewell.csv")
    }
    horizontal_ids = {
        file.name.replace("__horizontal_well.csv", "")
        for file in test_dir.rglob("*__horizontal_well.csv")
    }
    if not typewell_ids and not horizontal_ids:
        raise FileNotFoundError(
            f"No Kaggle test well files were found in {test_dir}."
        )
    missing_typewells = sorted(horizontal_ids - typewell_ids)
    missing_horizontal_wells = sorted(typewell_ids - horizontal_ids)
    if missing_typewells or missing_horizontal_wells:
        raise ValueError(
            "Incomplete Kaggle test well pairs. "
            f"Missing typewells: {missing_typewells}; "
            f"missing horizontal wells: {missing_horizontal_wells}."
        )
    return sorted(typewell_ids)


def _resolve_well_csv(directory: Path, filename: str) -> Path:
    direct = directory / filename
    if direct.exists():
        return direct
    matches = sorted(directory.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"{filename} not found under {directory}")
    return matches[0]


def load_horizontal_well(well_id: str, train_dir: Path) -> pd.DataFrame:
    return pd.read_csv(
        _resolve_well_csv(train_dir, f"{well_id}__horizontal_well.csv")
    )


def load_test_horizontal_well(well_id: str, test_dir: Path) -> pd.DataFrame:
    return pd.read_csv(
        _resolve_well_csv(test_dir, f"{well_id}__horizontal_well.csv")
    )


def create_horizontal_features(
    horizontal_df: pd.DataFrame,
    well_id: str,
) -> pd.DataFrame:
    if horizontal_df.empty:
        raise ValueError(f"Horizontal well {well_id} is empty.")

    required_columns = ["MD", "X", "Y", "Z", "GR", "TVT_input"]
    missing_columns = [
        column
        for column in required_columns
        if column not in horizontal_df.columns
    ]
    if missing_columns:
        raise ValueError(
            f"Well {well_id} is missing required columns: {missing_columns}"
        )

    df = horizontal_df.copy().sort_values("MD").reset_index(drop=True)
    df[GROUP_COLUMN] = well_id

    df["MD_relative"] = df["MD"] - df["MD"].iloc[0]
    df["X_relative"] = df["X"] - df["X"].iloc[0]
    df["Y_relative"] = df["Y"] - df["Y"].iloc[0]
    df["Z_relative"] = df["Z"] - df["Z"].iloc[0]

    df["MD_diff"] = df["MD"].diff()
    df["X_diff"] = df["X"].diff()
    df["Y_diff"] = df["Y"].diff()
    df["Z_diff"] = df["Z"].diff()
    df["GR_diff"] = df["GR"].diff()
    df["TVT_input_diff"] = df["TVT_input"].diff()

    df["horizontal_distance"] = np.sqrt(
        df["X_relative"] ** 2 + df["Y_relative"] ** 2
    )
    df["spatial_distance"] = np.sqrt(
        df["X_relative"] ** 2
        + df["Y_relative"] ** 2
        + df["Z_relative"] ** 2
    )
    return df


def create_preprocessor(feature_columns, scale_features: bool = True):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_features:
        from sklearn.preprocessing import StandardScaler

        numeric_steps.append(("scaler", StandardScaler()))

    numeric_pipeline = Pipeline(steps=numeric_steps)
    return ColumnTransformer(
        transformers=[("numeric", numeric_pipeline, list(feature_columns))]
    )


def create_extra_trees_pipeline(feature_columns, model_params=None):
    preprocessor = create_preprocessor(
        feature_columns=feature_columns,
        scale_features=False,
    )
    default_params = {
        "n_estimators": 100,
        "max_depth": 18,
        "min_samples_leaf": 50,
        "bootstrap": True,
        "max_samples": 0.10,
        "random_state": 42,
        "n_jobs": 4,
    }
    if model_params is not None:
        default_params.update(model_params)
    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", ExtraTreesRegressor(**default_params)),
        ]
    )


def create_xgboost_pipeline(feature_columns, model_params=None):
    preprocessor = create_preprocessor(
        feature_columns=feature_columns,
        scale_features=False,
    )
    default_params = {
        "objective": "reg:squarederror",
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 8,
        "min_child_weight": 20,
        "subsample": 0.50,
        "colsample_bytree": 0.80,
        "reg_alpha": 0.10,
        "reg_lambda": 1.00,
        "tree_method": "hist",
        "max_bin": 256,
        "random_state": 42,
        "n_jobs": 4,
        "verbosity": 1,
    }
    if model_params is not None:
        default_params.update(model_params)
    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", XGBRegressor(**default_params)),
        ]
    )


print("Utilities ready.")


## 4. Final Model Constants


In [ ]:
MODEL_FEATURE_COLUMNS = [
    "MD",
    "GR",
    "TVT_input",
    "X",
    "Y",
    "Z",
]

EXTRA_TREES_WEIGHT = 0.80
XGBOOST_WEIGHT = 0.20

EXTRA_TREES_PARAMS = {
    "n_estimators": 150,
    "max_depth": 20,
    "min_samples_leaf": 25,
    "bootstrap": True,
    "max_samples": 0.2,
    "random_state": 42,
    "n_jobs": 4,
}

XGBOOST_PARAMS = {
    "n_estimators": 500,
}

assert np.isclose(EXTRA_TREES_WEIGHT + XGBOOST_WEIGHT, 1.0)

print("Model features:", MODEL_FEATURE_COLUMNS)
print("Extra Trees weight:", EXTRA_TREES_WEIGHT)
print("XGBoost weight:", XGBOOST_WEIGHT)
print("Extra Trees params:", EXTRA_TREES_PARAMS)
print("XGBoost overrides:", XGBOOST_PARAMS)


## 5. Load Sample Submission


In [ ]:
search_roots = [TEST_DIR, DATA_ROOT]
if DATA_ROOT.parent != DATA_ROOT:
    search_roots.append(DATA_ROOT.parent)

submission_candidates = sorted({
    path
    for root in search_roots
    if root.exists()
    for path in root.rglob("*submission*.csv")
})

assert submission_candidates, (
    "sample_submission.csv was not found near train/test."
)

sample_submission_path = submission_candidates[0]
sample_submission_df = pd.read_csv(sample_submission_path)

assert list(sample_submission_df.columns) == ["id", "tvt"], (
    f"Unexpected sample submission columns: {sample_submission_df.columns.tolist()}"
)
assert len(sample_submission_df) > 0

print("Sample submission:", sample_submission_path)
print("Shape:", sample_submission_df.shape)
display(sample_submission_df.head())


## 6. Build Full Training Dataset

Train on all horizontal-well rows with engineered features, then keep only the six columns available in both train and Kaggle test.


In [ ]:
training_well_ids = discover_training_wells(TRAIN_DIR)

assert training_well_ids, f"No training well pairs found in {TRAIN_DIR}"

print(f"Training well pairs: {len(training_well_ids):,}")

training_feature_frames = []
failed_training_wells = []
training_start = time.time()

for well_index, well_id in enumerate(training_well_ids, start=1):
    try:
        horizontal_df = load_horizontal_well(well_id, TRAIN_DIR)
        if "TVT" not in horizontal_df.columns:
            raise ValueError(f"Missing TVT target for well {well_id}")

        target_tvt = horizontal_df["TVT"].copy()
        well_features_df = create_horizontal_features(
            horizontal_df=horizontal_df,
            well_id=well_id,
        )

        if len(well_features_df) != len(target_tvt):
            raise ValueError(
                f"Feature-target row mismatch for well {well_id}"
            )

        well_features_df = well_features_df.copy()
        well_features_df["target_tvt"] = target_tvt.to_numpy()
        well_features_df["well_id"] = well_id
        training_feature_frames.append(well_features_df)
    except Exception as error:
        failed_training_wells.append({
            "well_id": well_id,
            "error": str(error),
        })

    if (
        well_index == 1
        or well_index % 50 == 0
        or well_index == len(training_well_ids)
    ):
        print(
            f"Processed {well_index:,} / {len(training_well_ids):,} | "
            f"ok={len(training_feature_frames):,} "
            f"failed={len(failed_training_wells):,}"
        )

training_seconds = time.time() - training_start

assert training_feature_frames, "No training wells were constructed successfully."
assert not failed_training_wells, (
    f"Failed training wells: {failed_training_wells[:5]}"
)

full_training_df = pd.concat(training_feature_frames, ignore_index=True)

missing_model_features = [
    column
    for column in MODEL_FEATURE_COLUMNS
    if column not in full_training_df.columns
]
assert not missing_model_features, missing_model_features

X_final = full_training_df[MODEL_FEATURE_COLUMNS].copy()
y_full = full_training_df["target_tvt"].copy()

print("Training rows:", f"{len(X_final):,}")
print("Training wells:", full_training_df["well_id"].nunique())
print("Construction seconds:", round(training_seconds, 2))
print("Feature matrix shape:", X_final.shape)


## 7. Train Final Production Models


In [ ]:
extra_trees_pipeline = create_extra_trees_pipeline(
    feature_columns=MODEL_FEATURE_COLUMNS,
    model_params=EXTRA_TREES_PARAMS,
)

print("Training Extra Trees...")
et_start = time.time()
extra_trees_pipeline.fit(X_final, y_full)
et_seconds = time.time() - et_start
print(f"Extra Trees trained in {et_seconds:.2f}s")

xgboost_pipeline = create_xgboost_pipeline(
    feature_columns=MODEL_FEATURE_COLUMNS,
    model_params=XGBOOST_PARAMS,
)

print("Training XGBoost...")
xgb_start = time.time()
xgboost_pipeline.fit(X_final, y_full)
xgb_seconds = time.time() - xgb_start
print(f"XGBoost trained in {xgb_seconds:.2f}s")

print(
    "Extra Trees n_estimators:",
    extra_trees_pipeline.named_steps["model"].get_params()["n_estimators"],
)
print(
    "XGBoost n_estimators:",
    xgboost_pipeline.named_steps["model"].get_params()["n_estimators"],
)


## 8. Construct Kaggle Test Features

Predict only rows where `TVT_input` is missing. Feature engineering runs on the full well first so relative/diff features stay consistent, then the prediction mask is applied.


In [ ]:
test_well_ids = discover_test_well_pairs(TEST_DIR)

horizontal_file_map = {
    path.name.replace("__horizontal_well.csv", ""): path
    for path in sorted(TEST_DIR.rglob("*__horizontal_well.csv"))
}
typewell_file_map = {
    path.name.replace("__typewell.csv", ""): path
    for path in sorted(TEST_DIR.rglob("*__typewell.csv"))
}

print("Test well pairs:", len(test_well_ids))
print("Test well IDs:", test_well_ids)

test_feature_frames = []
global_prediction_position = 0
test_start = time.time()

for well_order, well_id in enumerate(test_well_ids):
    horizontal_df = pd.read_csv(horizontal_file_map[well_id])
    original_row_count = len(horizontal_df)

    prediction_mask = horizontal_df["TVT_input"].isna()
    number_of_prediction_rows = int(prediction_mask.sum())

    engineered_horizontal_df = create_horizontal_features(
        horizontal_df.copy(),
        well_id=well_id,
    )
    assert len(engineered_horizontal_df) == original_row_count

    missing_columns = [
        column
        for column in MODEL_FEATURE_COLUMNS
        if column not in engineered_horizontal_df.columns
    ]
    assert not missing_columns, missing_columns

    well_prediction_features_df = (
        engineered_horizontal_df
        .loc[prediction_mask, MODEL_FEATURE_COLUMNS]
        .copy()
        .reset_index(drop=True)
    )
    assert len(well_prediction_features_df) == number_of_prediction_rows

    original_prediction_positions = np.flatnonzero(prediction_mask.to_numpy())

    well_prediction_features_df.insert(
        0, "original_row_position", original_prediction_positions
    )
    well_prediction_features_df.insert(0, "well_order", well_order)
    well_prediction_features_df.insert(0, "well_id", well_id)
    well_prediction_features_df["global_prediction_position"] = np.arange(
        global_prediction_position,
        global_prediction_position + number_of_prediction_rows,
    )
    global_prediction_position += number_of_prediction_rows
    test_feature_frames.append(well_prediction_features_df)

assert test_feature_frames, "No test prediction rows were constructed."

test_features_with_metadata_df = pd.concat(
    test_feature_frames, ignore_index=True
)
test_seconds = time.time() - test_start

assert len(test_features_with_metadata_df) == len(sample_submission_df), (
    "Constructed prediction rows do not match the sample submission.\n"
    f"Constructed: {len(test_features_with_metadata_df):,}\n"
    f"Expected: {len(sample_submission_df):,}"
)

X_test_final = test_features_with_metadata_df[MODEL_FEATURE_COLUMNS].copy()
test_metadata_df = test_features_with_metadata_df[
    [
        "well_id",
        "well_order",
        "original_row_position",
        "global_prediction_position",
    ]
].copy()

test_metadata_df.insert(0, "id", sample_submission_df["id"].to_numpy())

assert not test_metadata_df["id"].duplicated().any()

print("Test prediction rows:", f"{len(X_test_final):,}")
print("Construction seconds:", round(test_seconds, 2))
display(test_metadata_df.head())


## 9. Predict and Build Weighted Ensemble


In [ ]:
print("Predicting with Extra Trees...")
et_pred_start = time.time()
extra_trees_test_predictions = np.asarray(
    extra_trees_pipeline.predict(X_test_final),
    dtype=float,
)
et_pred_seconds = time.time() - et_pred_start

print("Predicting with XGBoost...")
xgb_pred_start = time.time()
xgboost_test_predictions = np.asarray(
    xgboost_pipeline.predict(X_test_final),
    dtype=float,
)
xgb_pred_seconds = time.time() - xgb_pred_start

assert len(extra_trees_test_predictions) == len(X_test_final)
assert len(xgboost_test_predictions) == len(X_test_final)
assert np.isfinite(extra_trees_test_predictions).all()
assert np.isfinite(xgboost_test_predictions).all()

weighted_ensemble_predictions = (
    EXTRA_TREES_WEIGHT * extra_trees_test_predictions
    + XGBOOST_WEIGHT * xgboost_test_predictions
)

assert np.isfinite(weighted_ensemble_predictions).all()
assert len(weighted_ensemble_predictions) == len(sample_submission_df)

print(f"Extra Trees predict seconds: {et_pred_seconds:.4f}")
print(f"XGBoost predict seconds: {xgb_pred_seconds:.4f}")
print("Ensemble min:", float(weighted_ensemble_predictions.min()))
print("Ensemble max:", float(weighted_ensemble_predictions.max()))
print("Ensemble mean:", float(weighted_ensemble_predictions.mean()))


## 10. Create and Validate Kaggle Submission


In [ ]:
submission_df = sample_submission_df.copy()
submission_df["tvt"] = weighted_ensemble_predictions

assert list(submission_df.columns) == ["id", "tvt"]
assert len(submission_df) == len(sample_submission_df)
assert np.array_equal(
    submission_df["id"].to_numpy(),
    sample_submission_df["id"].to_numpy(),
)
assert np.array_equal(
    submission_df["id"].to_numpy(),
    test_metadata_df["id"].to_numpy(),
)
assert submission_df["tvt"].notna().all()
assert np.isfinite(submission_df["tvt"].to_numpy(dtype=float)).all()
assert not submission_df["id"].duplicated().any()

FINAL_SUBMISSION_PATH = OUT_DIR / "submission.csv"
submission_df.to_csv(FINAL_SUBMISSION_PATH, index=False)

saved_submission_df = pd.read_csv(FINAL_SUBMISSION_PATH)

assert list(saved_submission_df.columns) == ["id", "tvt"]
assert len(saved_submission_df) == len(sample_submission_df)
assert np.array_equal(
    saved_submission_df["id"].to_numpy(),
    sample_submission_df["id"].to_numpy(),
)
assert np.allclose(
    saved_submission_df["tvt"].to_numpy(dtype=float),
    weighted_ensemble_predictions,
)

print("Submission saved to:", FINAL_SUBMISSION_PATH)
print("Rows:", len(saved_submission_df))
display(saved_submission_df.head())
display(saved_submission_df.tail())
print("Ready for Kaggle upload.")
